# Chapter 2 — Accord Combinations

Publication-oriented analysis, summary tables and visualizations for *Fragrance Accord Intelligence*.

This notebook prepares the Chapter 2 outputs for ordered accord dyads and triads. It reuses the source definitions and median-based segmentation logic established in the earlier analysis, while applying the shared publication design system.

## Figure and table inventory

| Item | Title |
|---|---|
| Figure 2.1 | Top 20 Ordered Accord Pairs |
| Figure 2.2 | Top 20 Ordered Accord Triads |
| Figure 2.3 | Dyad Attention–Rating Segmentation |
| Figure 2.4 | Triad Attention–Rating Segmentation |
| Figures 2.5–2.8 | Dyad Segment Profiles |
| Figures 2.9–2.12 | Triad Segment Profiles |
| Table 2.1 | Dyad Segment Summary |
| Table 2.2 | Triad Segment Summary |
| Table 2.3 | Dyad and Triad Segment Comparison |

## Analytical conventions

- Dyads preserve the first and second accord positions; reversed pairs remain distinct.
- Triads preserve the first, second and third accord positions.
- Dyad aggregation retains combinations with at least 30 fragrances; segmentation requires at least 100.
- Triad segmentation requires at least 50 fragrances.
- Attention is measured as mean community votes per fragrance.
- Ratings are platform-user evaluations, not objective quality measures.
- Segment thresholds are calculated independently for dyads and triads.
- Median ties are assigned to the higher side using the `>=` rule.
- Segment-specific charts rank combinations by the share of associated fragrances rated 4.5 or above. This is a secondary descriptive measure, not the basis for segment membership.
- Fragrance counts across combinations are assignments and should not be interpreted as unique fragrance totals.

## 0. Setup and source data

Load the cleaned Fragrantica source table and prepare the rated-fragrance dataset. Accord parsing and positional extraction follow the established Notebook 10 logic.

In [0]:
# Install the plotting libraries used by this notebook.
# Matplotlib is provided by the Databricks Python environment.
%pip install lets-plot
%pip install adjustText

In [0]:
# ============================================================
# NOTEBOOK 17 — CHAPTER 2 ACCORD COMBINATIONS
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import base64
import io

from lets_plot import (
    LetsPlot, ggplot, aes, geom_bar, geom_text, coord_flip,
    labs, theme, theme_minimal, element_text, element_line,
    element_rect, element_blank, scale_fill_manual, scale_y_continuous,
    ggsave
)

# Configure Lets-Plot for notebook rendering.
LetsPlot.setup_html()

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

print(f"Source table loaded: {SOURCE_TABLE}")


In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.25  # Reduced physical gap between the subtitle and chart body

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_BAR_SIZES = {
    "category_pt": 14,  # Larger accord/combinations labels
    "value_pt": 14      # Larger values displayed at bar ends
}

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "15px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Store table content and metadata for Matplotlib rendering."""
    return {
        "data": data.copy(),
        "title": title,
        "subtitle": subtitle
    }


def _render_book_table(table):
    """Render a publication-style table using Matplotlib only."""
    import textwrap

    data = table["data"].copy()
    n_rows, n_cols = data.shape

    # Give wider columns to longer headings and values.
    raw_widths = []
    for column in data.columns:
        samples = [str(column)] + data[column].fillna("").astype(str).tolist()
        max_chars = max((len(value) for value in samples), default=8)
        raw_widths.append(max(8, min(30, max_chars + 2)))

    width_total = sum(raw_widths)
    column_widths = [value / width_total for value in raw_widths]

    figure_width = max(10.5, min(24, 2.0 + n_cols * 1.55))
    figure_height = max(3.7, 2.45 + n_rows * 0.56)
    fig, ax = plt.subplots(figsize=(figure_width, figure_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])
    ax.axis("off")

    # Keep headings readable in wider tables by wrapping long labels.
    wrapped_headers = []
    for label, width in zip(data.columns, column_widths):
        wrap_width = max(9, int(width * 105))
        wrapped_headers.append(textwrap.fill(str(label), width=wrap_width))

    cell_text = data.fillna("").astype(str).values.tolist()
    table_artist = ax.table(
        cellText=cell_text,
        colLabels=wrapped_headers,
        colWidths=column_widths,
        cellLoc="left",
        colLoc="center",
        bbox=[0.025, 0.025, 0.95, 0.76]
    )
    table_artist.auto_set_font_size(False)

    body_font_size = 10.5 if n_cols <= 7 else 8.5
    header_font_size = 9.5 if n_cols <= 7 else 7.8

    for (row, col), cell in table_artist.get_celld().items():
        cell.set_edgecolor(BOOK_STYLE["rule"])
        cell.set_linewidth(0.65)
        cell.PAD = 0.12

        if row == 0:
            cell.set_facecolor("#40332E")
            cell.set_text_props(
                color="#FFFDF9",
                weight="bold",
                fontsize=header_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                ha="center",
                va="center"
            )
            cell.set_height(0.105 if n_cols <= 7 else 0.14)
        else:
            cell.set_facecolor("#FFFDF9" if row % 2 else "#F0E9DF")
            cell.set_text_props(
                color=BOOK_STYLE["text"],
                fontsize=body_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                va="center"
            )
            if col == 0:
                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    weight="bold",
                    fontsize=body_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="left",
                    va="center"
                )
            else:
                cell.set_text_props(ha="right")
            cell.set_height(0.085)

    fig.text(
        0.055, 0.955, table["title"],
        ha="left", va="top",
        color=BOOK_STYLE["text"],
        fontsize=20,
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold"
    )

    subtitle = table.get("subtitle")
    if subtitle:
        fig.text(
            0.057, 0.895, subtitle,
            ha="left", va="top",
            color=BOOK_STYLE["muted"],
            fontsize=10.5,
            fontfamily=BOOK_TYPOGRAPHY["body"]
        )

    return fig


def show_book_table(table):
    """Display the Matplotlib-rendered table in the notebook."""
    fig = _render_book_table(table)
    show_book_matplotlib(fig)
    plt.close(fig)


def save_book_table_png(table, filename):
    """Save a high-resolution Matplotlib table PNG to the Databricks Volume."""
    from pathlib import Path

    output_dir = Path(
        "/Volumes/main/fragrance_analytics/reference_data/visualizations"
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    temporary_path = output_dir / ("." + output_path.stem + ".rendering.png")

    fig = _render_book_table(table)
    try:
        fig.savefig(
            temporary_path,
            format="png",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.18,
            facecolor=BOOK_STYLE["background"],
            edgecolor=BOOK_STYLE["background"]
        )
    finally:
        plt.close(fig)

    assert temporary_path.is_file() and temporary_path.stat().st_size > 0, (
        f"Matplotlib did not create a valid table PNG: {temporary_path}"
    )
    temporary_path.replace(output_path)

    assert output_path.is_file() and output_path.stat().st_size > 0
    print(
        f"Saved Matplotlib table PNG: {output_path} "
        f"({output_path.stat().st_size:,} bytes)"
    )

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.25  # Reduced physical gap between the subtitle and chart body

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_BAR_SIZES = {
    "category_pt": 14,  # Larger accord/combinations labels
    "value_pt": 14      # Larger values displayed at bar ends
}

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "15px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Store table content and metadata for Matplotlib rendering."""
    return {
        "data": data.copy(),
        "title": title,
        "subtitle": subtitle
    }


def _render_book_table(table):
    """Render a publication-style table using Matplotlib only."""
    import textwrap

    data = table["data"].copy()
    n_rows, n_cols = data.shape

    # Give wider columns to longer headings and values.
    raw_widths = []
    for column in data.columns:
        samples = [str(column)] + data[column].fillna("").astype(str).tolist()
        max_chars = max((len(value) for value in samples), default=8)
        raw_widths.append(max(8, min(30, max_chars + 2)))

    width_total = sum(raw_widths)
    column_widths = [value / width_total for value in raw_widths]

    figure_width = max(10.5, min(24, 2.0 + n_cols * 1.55))
    figure_height = max(3.7, 2.45 + n_rows * 0.56)
    fig, ax = plt.subplots(figsize=(figure_width, figure_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])
    # Use the full figure canvas so table and heading coordinates share one origin.
    ax.set_position([0, 0, 1, 1])
    ax.axis("off")

    # Keep headings readable in wider tables by wrapping long labels.
    wrapped_headers = []
    for label, width in zip(data.columns, column_widths):
        wrap_width = max(9, int(width * 105))
        wrapped_headers.append(textwrap.fill(str(label), width=wrap_width))

    cell_text = data.fillna("").astype(str).values.tolist()
    table_artist = ax.table(
        cellText=cell_text,
        colLabels=wrapped_headers,
        colWidths=column_widths,
        cellLoc="left",
        colLoc="center",
        bbox=[0.055, 0.025, 0.92, 0.76]
    )
    table_artist.auto_set_font_size(False)

    body_font_size = 12.5 if n_cols <= 7 else 10.5
    header_font_size = 9.5 if n_cols <= 7 else 7.8

    for (row, col), cell in table_artist.get_celld().items():
        cell.set_edgecolor(BOOK_STYLE["rule"])
        cell.set_linewidth(0.65)
        cell.PAD = 0.12

        if row == 0:
            cell.set_facecolor("#40332E")
            cell.set_text_props(
                color="#FFFDF9",
                weight="bold",
                fontsize=header_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                ha="center",
                va="center"
            )
            cell.set_height(0.105 if n_cols <= 7 else 0.14)
        else:
            cell.set_facecolor("#FFFDF9" if row % 2 else "#F0E9DF")
            cell.set_text_props(
                color=BOOK_STYLE["text"],
                fontsize=body_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                va="center"
            )
            if col == 0:
                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    weight="bold",
                    fontsize=body_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="left",
                    va="center"
                )
            else:
                cell.set_text_props(ha="right")
            cell.set_height(0.085)

    fig.text(
        0.055, 0.955, table["title"],
        ha="left", va="top",
        color=BOOK_STYLE["text"],
        fontsize=20,
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold"
    )

    subtitle = table.get("subtitle")
    if subtitle:
        fig.text(
            0.057, 0.895, subtitle,
            ha="left", va="top",
            color=BOOK_STYLE["muted"],
            fontsize=10.5,
            fontfamily=BOOK_TYPOGRAPHY["body"]
        )

    return fig


def show_book_table(table):
    """Display the Matplotlib-rendered table in the notebook."""
    fig = _render_book_table(table)
    show_book_matplotlib(fig)
    plt.close(fig)


def save_book_table_png(table, filename):
    """Save a high-resolution Matplotlib table PNG to the Databricks Volume."""
    from pathlib import Path

    output_dir = Path(
        "/Volumes/main/fragrance_analytics/reference_data/visualizations"
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    temporary_path = output_dir / ("." + output_path.stem + ".rendering.png")

    fig = _render_book_table(table)
    try:
        fig.savefig(
            temporary_path,
            format="png",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.18,
            facecolor=BOOK_STYLE["background"],
            edgecolor=BOOK_STYLE["background"]
        )
    finally:
        plt.close(fig)

    assert temporary_path.is_file() and temporary_path.stat().st_size > 0, (
        f"Matplotlib did not create a valid table PNG: {temporary_path}"
    )
    temporary_path.replace(output_path)

    assert output_path.is_file() and output_path.stat().st_size > 0
    print(
        f"Saved Matplotlib table PNG: {output_path} "
        f"({output_path.stat().st_size:,} bytes)"
    )

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

## 0.2 Shared rated-fragrance accord data

Prepare the fragrance-level dataset once for both dyad and triad analyses. Preserve the original first, second and third accord positions, and validate the one-row-per-fragrance assumption before aggregation.

In [0]:
# ============================================================
# CHAPTER 2 — SHARED RATED-FRAGRANCE ACCORD DATA
# Shared by ordered dyad and triad analysis.
# ============================================================

from pathlib import Path
from pyspark.sql import functions as F

CHAPTER_2_OUTPUT_DIR = Path(
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
CHAPTER_2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Recreate the valid-accord flag using Notebook 10's definition.
accords_clean_ch2 = F.trim(F.col("main_accords"))

valid_accords_ch2 = (
    F.col("main_accords").isNotNull()
    & (accords_clean_ch2 != "")
    & (~F.lower(accords_clean_ch2).isin("null", "none", "nan", "n/a", "[]"))
)

fragrantica_rating_base_ch2 = df_perfumes.withColumn(
    "has_valid_accords",
    valid_accords_ch2
)

# Parse accord arrays while preserving their original order.
accord_array_ch2 = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\\bNone\\b",
        "null"
    ),
    "array<string>"
)

rating_accords_ch2 = (
    fragrantica_rating_base_ch2
    .filter(
        F.col("has_valid_accords")
        & F.col("rating_value").isNotNull()
    )
    .withColumn("accords", accord_array_ch2)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
)

# Extract first, second and third accord positions without reordering.
for position, column_name in [
    (0, "dominant_accord"),
    (1, "second_accord"),
    (2, "third_accord")
]:
    rating_accords_ch2 = rating_accords_ch2.withColumn(
        column_name,
        F.when(
            (F.size("accords") > position)
            & F.col("accords")[position].isNotNull()
            & (F.trim(F.col("accords")[position]) != ""),
            F.col("accords")[position]
        )
    )

rating_accords_ch2 = rating_accords_ch2.select(
    "id",
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "dominant_accord",
    "second_accord",
    "third_accord"
)

# Validate the one-row-per-fragrance assumption before either aggregation.
source_rows_ch2 = rating_accords_ch2.count()
distinct_ids_ch2 = rating_accords_ch2.select("id").distinct().count()
assert source_rows_ch2 == distinct_ids_ch2, (
    f"Duplicate fragrance records detected: {source_rows_ch2:,} rows "
    f"versus {distinct_ids_ch2:,} unique IDs."
)
print(f"Rated fragrances with valid accord data: {source_rows_ch2:,}")
print(f"Distinct fragrance IDs: {distinct_ids_ch2:,}")
print(f"Chapter 2 output directory: {CHAPTER_2_OUTPUT_DIR}")

## 2.1 Dyad aggregation

Reproduce the ordered pair extraction and aggregation from Notebook 10 using the same fragrance-level source and first/second accord positions. Objects use a `_ch2` suffix to avoid collisions.

In [0]:
# ============================================================
# CHAPTER 2 — ORDERED DYAD DATA PREPARATION
# ============================================================

MIN_PAIR_FRAGRANCES_CH2 = 30
MIN_SEGMENT_PAIR_FRAGRANCES_CH2 = 100

accord_pair_stats_ch2 = (
    rating_accords_ch2
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr("percentile_approx(rating_count, 0.5)").alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr("percentile_approx(rating_value, 0.5)").alias("median_rating"),
        F.avg(F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)).alias("share_rated_4_5_plus")
    )
    .filter(F.col("fragrance_count") >= MIN_PAIR_FRAGRANCES_CH2)
    .withColumn("accord_pair", F.concat_ws(" + ", F.initcap("dominant_accord"), F.initcap("second_accord")))
)

eligible_pairs_ch2 = accord_pair_stats_ch2.filter(
    F.col("fragrance_count") >= MIN_SEGMENT_PAIR_FRAGRANCES_CH2
)

print(f"Ordered pairs with n >= {MIN_PAIR_FRAGRANCES_CH2}: {accord_pair_stats_ch2.count():,}")
print(f"Eligible ordered pairs for segmentation (n >= {MIN_SEGMENT_PAIR_FRAGRANCES_CH2}): {eligible_pairs_ch2.count():,}")

display(
    eligible_pairs_ch2.orderBy(F.desc("fragrance_count"), F.desc("mean_rating"))
    .select("accord_pair", "fragrance_count", "mean_votes", "mean_rating", "share_rated_4_5_plus")
    .limit(20)
)

## 2.2 Table 2.1 and dyad segmentation validation

The eligible dyad population is validated before the median thresholds and segment summaries are calculated.

In [0]:
# ============================================================
# TABLE 2.1 — ORDERED DYAD SEGMENTATION SUMMARY
# Segments as columns, measures as rows
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path
import os
import tempfile

# ------------------------------------------------------------
# 1. Prepare dyad data
# ------------------------------------------------------------

pairs_pdf_ch2 = eligible_pairs_ch2.toPandas().copy()

numeric_columns_ch2 = [
    "fragrance_count",
    "mean_votes",
    "mean_rating",
    "share_rated_4_5_plus"
]

for column in numeric_columns_ch2:
    pairs_pdf_ch2[column] = pd.to_numeric(
        pairs_pdf_ch2[column],
        errors="coerce"
    )

pairs_pdf_ch2 = pairs_pdf_ch2.dropna(
    subset=["accord_pair"] + numeric_columns_ch2
).copy()

assert not pairs_pdf_ch2.empty, (
    "No eligible ordered dyads are available for Table 2.1."
)

# ------------------------------------------------------------
# 2. Calculate median thresholds and assign segments
# ------------------------------------------------------------

median_attention_pairs_ch2 = pairs_pdf_ch2["mean_votes"].median()
median_rating_pairs_ch2 = pairs_pdf_ch2["mean_rating"].median()

pairs_pdf_ch2["segment"] = np.select(
    [
        (pairs_pdf_ch2["mean_votes"] >= median_attention_pairs_ch2) &
        (pairs_pdf_ch2["mean_rating"] >= median_rating_pairs_ch2),

        (pairs_pdf_ch2["mean_votes"] < median_attention_pairs_ch2) &
        (pairs_pdf_ch2["mean_rating"] >= median_rating_pairs_ch2),

        (pairs_pdf_ch2["mean_votes"] >= median_attention_pairs_ch2) &
        (pairs_pdf_ch2["mean_rating"] < median_rating_pairs_ch2)
    ],
    [
        "Standout Profiles",
        "Hidden Favourites",
        "Widely Explored"
    ],
    default="Quiet Profiles"
)

segment_order_ch2 = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

# ------------------------------------------------------------
# 3. Aggregate segment metrics
# ------------------------------------------------------------

pair_segment_summary_ch2 = (
    pairs_pdf_ch2
    .groupby("segment")
    .agg(
        pair_count=("accord_pair", "count"),
        fragrance_assignments=("fragrance_count", "sum"),
        median_fragrance_count=("fragrance_count", "median"),
        mean_votes_per_pair=("mean_votes", "mean"),
        mean_rating_per_pair=("mean_rating", "mean"),
        mean_pair_share_rated_4_5_plus=("share_rated_4_5_plus", "mean")
    )
    .reindex(segment_order_ch2)
)

# ------------------------------------------------------------
# 4. Build transposed display table
# ------------------------------------------------------------

pair_segment_display_ch2 = pair_segment_summary_ch2[
    [
        "pair_count",
        "fragrance_assignments",
        "median_fragrance_count",
        "mean_votes_per_pair",
        "mean_rating_per_pair",
        "mean_pair_share_rated_4_5_plus"
    ]
].T.copy()

pair_segment_display_ch2.index = [
    "Number of ordered dyads",
    "Fragrance assignments",
    "Median fragrances per dyad",
    "Mean votes per dyad",
    "Mean rating per dyad",
    "Share rated 4.5+"
]

# ------------------------------------------------------------
# 5. Format values
# ------------------------------------------------------------

for segment in segment_order_ch2:

    if segment not in pair_segment_display_ch2.columns:
        continue

    values = pair_segment_display_ch2[segment]

    pair_segment_display_ch2[segment] = [
        f"{values.iloc[0]:,.0f}" if pd.notna(values.iloc[0]) else "—",
        f"{values.iloc[1]:,.0f}" if pd.notna(values.iloc[1]) else "—",
        f"{values.iloc[2]:,.1f}" if pd.notna(values.iloc[2]) else "—",
        f"{values.iloc[3]:,.1f}" if pd.notna(values.iloc[3]) else "—",
        f"{values.iloc[4]:.3f}" if pd.notna(values.iloc[4]) else "—",
        f"{values.iloc[5]:.1%}" if pd.notna(values.iloc[5]) else "—"
    ]

pair_segment_display_ch2 = (
    pair_segment_display_ch2
    .reset_index()
    .rename(columns={"index": "Measure"})
)

# ------------------------------------------------------------
# 6. Dedicated Matplotlib renderer
# ------------------------------------------------------------

def render_pair_segment_table_ch2(table):

    data = table["data"].copy()

    fig, ax = plt.subplots(
        figsize=(15, 7),
        dpi=180
    )

    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    ax.set_position([0, 0, 1, 1])
    ax.axis("off")

    table_artist = ax.table(
        cellText=data.fillna("").astype(str).values.tolist(),
        colLabels=data.columns.tolist(),

        # Measure = 33%; four segment columns share 67%
        colWidths=[0.33] + [0.1675] * (len(data.columns) - 1),

        cellLoc="center",
        colLoc="center",
        bbox=[0.055, 0.10, 0.92, 0.69]
    )

    table_artist.auto_set_font_size(False)

    # Matching font size for column headers and table body
    matching_font_size = 15

    for (row, col), cell in table_artist.get_celld().items():

        cell.set_edgecolor(BOOK_STYLE["rule"])
        cell.set_linewidth(0.75)
        cell.PAD = 0.055

        if row == 0:

            cell.set_facecolor("#40332E")

            cell.set_text_props(
                color="#FFFDF9",
                weight="bold",
                fontsize=matching_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                ha="center",
                va="center"
            )

            cell.set_height(0.14)

        else:

            cell.set_facecolor(
                "#FFFDF9" if row % 2 else "#F0E9DF"
            )

            cell.set_height(0.12)

            if col == 0:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    weight="bold",
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="left",
                    va="center"
                )

            else:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="center",
                    va="center"
                )

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    fig.text(
        0.055,
        0.94,
        table["title"],
        color=BOOK_STYLE["text"],
        fontsize=20,
        fontweight="bold",
        ha="left",
        va="top"
    )

    # --------------------------------------------------------
    # Subtitle
    # --------------------------------------------------------

    subtitle = table.get("subtitle")

    if subtitle:

        fig.text(
            0.057,
            0.875,
            subtitle,
            color=BOOK_STYLE["muted"],
            fontsize=15,
            fontfamily=BOOK_TYPOGRAPHY["body"],
            ha="left",
            va="top"
        )

    return fig


# ------------------------------------------------------------
# 7. Display function
# ------------------------------------------------------------

def show_pair_segment_table_ch2(table):

    fig = render_pair_segment_table_ch2(table)

    show_book_matplotlib(fig)

    plt.close(fig)


# ------------------------------------------------------------
# 8. PNG export function
# ------------------------------------------------------------

def save_pair_segment_table_ch2(table, filename):

    output_dir = Path(
        "/Volumes/main/fragrance_analytics/reference_data/visualizations"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = output_dir / filename
    temp_path = None

    fig = render_pair_segment_table_ch2(table)

    try:

        with tempfile.NamedTemporaryFile(
            suffix=".png",
            dir=output_dir,
            delete=False
        ) as temp_file:

            temp_path = Path(temp_file.name)

        fig.savefig(
            temp_path,
            dpi=300,
            bbox_inches=None,
            pad_inches=0,
            facecolor=fig.get_facecolor()
        )

        os.replace(
            temp_path,
            output_path
        )

    finally:

        plt.close(fig)

        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

    print(f"Saved: {output_path}")


# ------------------------------------------------------------
# 9. Create, display and export Table 2.1
# ------------------------------------------------------------

pair_segment_table_ch2 = make_book_table(
    pair_segment_display_ch2,
    title="Ordered Dyad Segmentation Summary",
    subtitle=(
        "Dyads classified independently using median attention "
        "and rating thresholds"
    )
)

show_pair_segment_table_ch2(pair_segment_table_ch2)

save_pair_segment_table_ch2(
    pair_segment_table_ch2,
    "table_2_1_ordered_dyad_segmentation_summary.png"
)

# ------------------------------------------------------------
# 10. Display top 20 ordered dyads
# ------------------------------------------------------------

display(
    pairs_pdf_ch2
    .sort_values(
        ["mean_votes", "mean_rating"],
        ascending=[False, False]
    )
    [[
        "accord_pair",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus",
        "segment"
    ]]
    .head(20)
)

## 2.3 Figure 2.1 — Most Frequent Ordered Accord Dyads

The chart ranks the 20 most frequent ordered dyads by the number of fragrances assigned to each combination. Accord order is retained in every label.

In [0]:
# ============================================================
# FIGURE 2.1 — MOST FREQUENT ORDERED DYADS
# ============================================================

def format_pair_label_ch2(value):
    return str(value)

plot_pairs_frequency_ch2 = (
    pairs_pdf_ch2.nlargest(20, "fragrance_count")
    .sort_values("fragrance_count", ascending=True).copy()
)
fig_2_1_dyads, ax = plt.subplots(figsize=(15, 10), dpi=180)
fig_2_1_dyads.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
y = np.arange(len(plot_pairs_frequency_ch2))
bars = ax.barh(y, plot_pairs_frequency_ch2["fragrance_count"],
               color=BOOK_STYLE["accent"], edgecolor="none", height=0.68, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_pairs_frequency_ch2["accord_pair"],
                   fontsize=BOOK_BAR_SIZES["category_pt"], color=BOOK_STYLE["text"])
max_value = float(plot_pairs_frequency_ch2["fragrance_count"].max())
for bar, value in zip(bars, plot_pairs_frequency_ch2["fragrance_count"]):
    ax.text(float(value) + max_value * 0.012, bar.get_y() + bar.get_height()/2,
            f"{int(value):,}", va="center", ha="left",
            fontsize=BOOK_BAR_SIZES["value_pt"], color=BOOK_STYLE["text"])
ax.set_xlim(0, max_value * 1.14)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="y", length=0, labelsize=BOOK_BAR_SIZES["category_pt"], pad=7)
ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["category_pt"], pad=7)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
apply_book_bar_layout(fig_2_1_dyads, bottom=0.08, right=0.96)
add_book_figure_header(
    fig_2_1_dyads,
    "The Most Frequent Ordered Accord Dyads",
    "Twenty most frequent combinations; n denotes the number of fragrances",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
fig_2_1_dyads.savefig(
    CHAPTER_2_OUTPUT_DIR / "figure_2_1_top_20_ordered_dyads.png",
    dpi=300, bbox_inches=None, facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_2_1_dyads)
plt.close(fig_2_1_dyads)
print(f"Saved: {CHAPTER_2_OUTPUT_DIR / 'figure_2_1_top_20_ordered_dyads.png'}")

## 2.4 Figure 2.3 — Community Attention and Mean Rating Across Ordered Dyads

The bubble chart positions all 147 eligible ordered dyads by mean community votes per fragrance and mean fragrance rating. Bubble size represents the number of fragrances in the dyad. Dashed reference lines mark medians across the eligible pair population. Up to 35 labels are selected for readability; they do not define the analytical population.

In [0]:
# ============================================================
# FIGURE 2.3 — ORDERED DYAD ATTENTION / RATING SEGMENTATION
# All eligible dyads are plotted; selected dyads are labelled.
# ============================================================

from matplotlib.lines import Line2D
from adjustText import adjust_text

N_LARGEST_DYAD_CH2 = 8
N_EXTREMES_DYAD_CH2 = 6
MAX_LABELS_DYAD_CH2 = 24
MIN_BUBBLE_DYAD_CH2 = 55
MAX_BUBBLE_DYAD_CH2 = 950

# ------------------------------------------------------------
# 1. CALCULATE BUBBLE SIZES
# ------------------------------------------------------------

dyad_counts_ch2 = pairs_pdf_ch2["fragrance_count"].astype(float)
dyad_count_min_ch2 = dyad_counts_ch2.min()
dyad_count_max_ch2 = dyad_counts_ch2.max()

def dyad_bubble_size_ch2(count):
    if dyad_count_max_ch2 == dyad_count_min_ch2:
        return MIN_BUBBLE_DYAD_CH2
    normalized = (
        (count - dyad_count_min_ch2)
        / (dyad_count_max_ch2 - dyad_count_min_ch2)
    )
    normalized = max(0, min(1, normalized))
    return (
        MIN_BUBBLE_DYAD_CH2
        + normalized ** 0.5
        * (MAX_BUBBLE_DYAD_CH2 - MIN_BUBBLE_DYAD_CH2)
    )

pairs_pdf_ch2["bubble_size"] = dyad_counts_ch2.apply(dyad_bubble_size_ch2)

# ------------------------------------------------------------
# 2. SELECT DYADS TO LABEL
# ------------------------------------------------------------

largest_dyads_ch2 = pairs_pdf_ch2.nlargest(
    N_LARGEST_DYAD_CH2, "fragrance_count"
)

dyad_labels_ch2 = pd.concat([
    pairs_pdf_ch2.nlargest(N_EXTREMES_DYAD_CH2, "mean_votes"),
    pairs_pdf_ch2.nsmallest(N_EXTREMES_DYAD_CH2, "mean_votes"),
    pairs_pdf_ch2.nlargest(N_EXTREMES_DYAD_CH2, "mean_rating"),
    pairs_pdf_ch2.nsmallest(N_EXTREMES_DYAD_CH2, "mean_rating"),
    largest_dyads_ch2
]).drop_duplicates("accord_pair").copy()

dyad_labels_ch2["is_largest"] = dyad_labels_ch2["accord_pair"].isin(
    largest_dyads_ch2["accord_pair"]
)
dyad_labels_ch2["attention_extremeness"] = (
    (dyad_labels_ch2["mean_votes"] - popularity_median_pairs_ch2).abs()
    / max(abs(popularity_median_pairs_ch2), 1)
)
dyad_labels_ch2["rating_extremeness"] = (
    dyad_labels_ch2["mean_rating"] - rating_median_pairs_ch2
).abs()

dyad_labels_ch2 = (
    dyad_labels_ch2
    .sort_values(
        ["is_largest", "attention_extremeness", "rating_extremeness"],
        ascending=[False, False, False]
    )
    .head(MAX_LABELS_DYAD_CH2)
)

# ------------------------------------------------------------
# 3. DEFINE CHAPTER 1 SEGMENT COLOURS
# ------------------------------------------------------------

segment_colors_pairs_ch2 = {
    "Standout Profiles": "#A66A4E",
    "Hidden Favourites": "#B58A9A",
    "Widely Explored": "#71818A",
    "Quiet Profiles": "#C5A85C"
}

# ------------------------------------------------------------
# 4. CREATE FIGURE
# ------------------------------------------------------------

fig_2_3_dyads, ax = plt.subplots(figsize=(12, 10), dpi=180)
fig_2_3_dyads.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

# ------------------------------------------------------------
# 5. PLOT ALL ELIGIBLE DYADS
# ------------------------------------------------------------

for segment_name in segment_order_pairs_ch2:
    subset = pairs_pdf_ch2[pairs_pdf_ch2["segment"] == segment_name]
    if not subset.empty:
        ax.scatter(
            subset["mean_votes"],
            subset["mean_rating"],
            s=subset["bubble_size"],
            color=segment_colors_pairs_ch2[segment_name],
            alpha=0.78,
            edgecolors=BOOK_STYLE["background"],
            linewidths=0.8,
            zorder=3
        )

# ------------------------------------------------------------
# 6. ADD DYAD LABELS
# ------------------------------------------------------------

texts = []
for _, row in dyad_labels_ch2.iterrows():
    label = (
        f"{row['accord_pair']} "
        f"(n={row['fragrance_count']:,.0f})"
    )
    text = ax.text(
        row["mean_votes"],
        row["mean_rating"],
        label,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["category_pt"] + 1,
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        alpha=0.98,
        zorder=5,
        bbox={
            "facecolor": BOOK_STYLE["background"],
            "edgecolor": "none",
            "alpha": 0.85,
            "pad": 1.5
        }
    )
    texts.append(text)

if texts:
    adjust_text(
        texts,
        ax=ax,
        expand=(1.2, 1.4),
        force_text=(0.5, 0.8),
        force_static=(0.5, 0.8),
        force_pull=(0.05, 0.1),
        ensure_inside_axes=True,
        iter_lim=500,
        arrowprops=dict(
            arrowstyle="-",
            color=BOOK_STYLE["muted"],
            lw=0.7,
            alpha=0.7
        )
    )

# ------------------------------------------------------------
# 7. ADD MEDIAN REFERENCE LINES
# ------------------------------------------------------------

ax.axvline(
    popularity_median_pairs_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)
ax.axhline(
    rating_median_pairs_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)

# ------------------------------------------------------------
# 8. FORMAT AXES AND GRID
# ------------------------------------------------------------

ax.set_xlabel(
    "Mean community votes per fragrance",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)
ax.set_ylabel(
    "Mean fragrance rating",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f"{y:.2f}"))
ax.tick_params(
    axis="both",
    length=0,
    labelsize=BOOK_SIZES["footer_pt"] + 1,
    pad=8,
    colors=BOOK_STYLE["text"]
)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="both",
    color=BOOK_STYLE["grid"],
    linewidth=0.7,
    alpha=0.9,
    zorder=0
)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# 9. LEGEND — RELATIVE SEGMENTS ONLY
# ------------------------------------------------------------

segment_handles_pairs_ch2 = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=9,
        markerfacecolor=segment_colors_pairs_ch2[name],
        markeredgecolor=BOOK_STYLE["background"],
        markeredgewidth=0.8,
        label=name
    )
    for name in segment_order_pairs_ch2
]

segment_legend_pairs_ch2 = ax.legend(
    handles=segment_handles_pairs_ch2,
    title="Relative segment",
    loc="lower right",
    bbox_to_anchor=(1.0, 0.0),
    frameon=True,
    facecolor=BOOK_STYLE["background"],
    edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["category_pt"] + 1,
    title_fontsize=BOOK_SIZES["category_pt"] + 2,
    labelcolor=BOOK_STYLE["text"],
    borderpad=0.9,
    labelspacing=0.75,
    handletextpad=0.8
)
segment_legend_pairs_ch2.get_title().set_color(BOOK_STYLE["muted"])
segment_legend_pairs_ch2.get_title().set_fontfamily(BOOK_TYPOGRAPHY["body"])

# ------------------------------------------------------------
# 10. ADD FIGURE HEADER AND APPLY LAYOUT
# ------------------------------------------------------------

add_book_figure_header(
    fig_2_3_dyads,
    "Ordered Accord Dyads: Community Attention and Mean Rating",
    "All eligible dyads  |  Bubble size: number of fragrances  |  "
    "Labels show dyad and fragrance count.",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)

fig_2_3_dyads.subplots_adjust(
    left=0.085,
    right=0.985,
    top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_2_3_dyads.get_figheight(),
    bottom=0.11
)

# ------------------------------------------------------------
# 11. RENDER AND SAVE
# ------------------------------------------------------------

fig_2_3_dyads.savefig(
    CHAPTER_2_OUTPUT_DIR / "figure_2_3_dyad_attention_rating_segmentation.png",
    dpi=300,
    bbox_inches=None,
    facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_2_3_dyads)
plt.close(fig_2_3_dyads)

print(f"Eligible dyads plotted: {len(pairs_pdf_ch2):,}")
print(f"Dyads labelled: {len(dyad_labels_ch2):,}")
print(f"Attention median: {popularity_median_pairs_ch2:.2f}")
print(f"Rating median: {rating_median_pairs_ch2:.3f}")
print(f"Saved: {CHAPTER_2_OUTPUT_DIR / 'figure_2_3_dyad_attention_rating_segmentation.png'}")


## 2.5 Figures 2.5–2.8 — Highest-Rated Share Within Each Dyad Segment

Each chart shows the 20 dyads within one segment with the highest share of fragrances rated 4.5 or above. This metric complements mean rating; it is not interchangeable with it. Bar labels include the percentage and the number of fragrances in the dyad.

In [0]:
# ============================================================
# FIGURES 2.5–2.8 — TOP DYADS BY HIGH-RATING SHARE
# ============================================================

def plot_pair_segment_ch2(segment_name, top_n=20):
    plot_df = (
        pairs_pdf_ch2[pairs_pdf_ch2["segment"] == segment_name]
        .nlargest(top_n,"share_rated_4_5_plus")
        .sort_values("share_rated_4_5_plus",ascending=True).copy()
    )
    if plot_df.empty:
        print(f"No dyads found for segment: {segment_name}")
        return

    fig_height=max(8,len(plot_df)*0.43+2.5)
    fig,ax=plt.subplots(figsize=(15,fig_height),dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])
    y=np.arange(len(plot_df))
    bars=ax.barh(y,plot_df["share_rated_4_5_plus"],
        color=segment_colors_pairs_ch2[segment_name],edgecolor="none",height=0.68,zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["accord_pair"],fontsize=BOOK_BAR_SIZES["category_pt"],color=BOOK_STYLE["text"])
    max_value=float(plot_df["share_rated_4_5_plus"].max())
    for bar,(_,row) in zip(bars,plot_df.iterrows()):
        value=float(row["share_rated_4_5_plus"])
        ax.text(value+max(value*0.012,0.001),bar.get_y()+bar.get_height()/2,
            f"{value:.1%}  (n={int(row['fragrance_count']):,})",
            va="center",ha="left",fontsize=BOOK_BAR_SIZES["value_pt"],color=BOOK_STYLE["text"])
    ax.set_xlim(0,max_value*1.42 if max_value>0 else 1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Share of fragrances rated 4.5+",fontsize=BOOK_SIZES["category_pt"],
        color=BOOK_STYLE["muted"],labelpad=10)
    ax.set_ylabel("")
    ax.tick_params(axis="y",length=0,labelsize=BOOK_BAR_SIZES["category_pt"],pad=7)
    ax.tick_params(axis="x",length=0,labelsize=BOOK_SIZES["footer_pt"],pad=7)
    ax.grid(axis="x",color=BOOK_STYLE["grid"],linewidth=0.8,zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    apply_book_bar_layout(fig,bottom=0.08,right=0.96)
    add_book_figure_header(fig,
        f"{segment_name}: Dyads with the Highest 4.5+ Share",
        "Top 20 ordered dyads by share of fragrances rated 4.5 or above",
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2")
    figure_no = {"Standout Profiles": 5, "Hidden Favourites": 6, "Widely Explored": 7, "Quiet Profiles": 8}[segment_name]
    filename = f"figure_2_{figure_no}_dyad_{segment_name.lower().replace(chr(32), chr(95))}.png"
    fig.savefig(CHAPTER_2_OUTPUT_DIR/filename,dpi=300,bbox_inches=None,
        facecolor=BOOK_STYLE["background"])
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR/filename}")

for segment_name in segment_order_pairs_ch2:
    plot_pair_segment_ch2(segment_name,top_n=20)

## 2.6 Ordered Accord Triads

This section prepares publication-ready visualizations of ordered three-accord fragrance profiles. It follows the source ordering and aggregation logic established in Notebook 10 while applying the shared ivory editorial design system defined in this notebook.

**Analytical conventions**
- The first, second and third positions are preserved; triads with different accord order remain distinct.
- The analysis includes rated fragrances with valid accord data and non-null first, second and third accords.
- The minimum support threshold is 50 fragrances per ordered triad.
- Mean community votes per fragrance are used as an engagement proxy, not as sales or market share.
- The four segments are assigned using the medians of mean votes and mean ratings across all eligible triads.
- The 35 bubble-chart annotations are selected for readability only; all eligible triads remain in the chart and segmentation.
- The share of fragrances rated 4.5+ is a separate descriptive measure from mean rating.

## 2.7 Triad aggregation

Using the shared rated-fragrance dataset prepared in Section 2.0, aggregate the first three accord positions into ordered triads. Apply the same metrics and minimum-support rule as Notebook 10.

In [0]:
# ============================================================
# CHAPTER 2 — ORDERED TRIAD AGGREGATION
# Shared fragrance-level source is prepared before the dyad section.
# ============================================================

# Aggregate ordered triads using the same metrics as Notebook 10.
accord_triad_stats_ch2 = (
    rating_accords_ch2
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy(
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr("percentile_approx(rating_count, 0.5)").alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr("percentile_approx(rating_value, 0.5)").alias("median_rating"),
        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
    .withColumn(
        "accord_triad",
        F.concat_ws(
            " + ",
            F.initcap("dominant_accord"),
            F.initcap("second_accord"),
            F.initcap("third_accord")
        )
    )
)

MIN_TRIAD_FRAGRANCES_CH2 = 50

eligible_triads_ch2 = accord_triad_stats_ch2.filter(
    F.col("fragrance_count") >= MIN_TRIAD_FRAGRANCES_CH2
)

print(f"Rated fragrances with valid accord data: {rating_accords_ch2.count():,}")
print(f"Distinct fragrance IDs: {rating_accords_ch2.select('id').distinct().count():,}")
print(f"Unique ordered triads before support filter: {accord_triad_stats_ch2.count():,}")
print(f"Eligible ordered triads (n >= {MIN_TRIAD_FRAGRANCES_CH2}): {eligible_triads_ch2.count():,}")

display(
    eligible_triads_ch2
    .orderBy(F.desc("fragrance_count"), F.desc("mean_rating"))
    .select(
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    )
    .limit(20)
)

## 2.8 Table 2.2 and triad segmentation validation

Validate the fragrance-level uniqueness and eligible triad population before generating publication figures. Segment summary means are calculated across triads, with each triad contributing equally; they are not fragrance-weighted averages.

In [0]:
# ============================================================
# CHAPTER 2 — VALIDATION AND SEGMENT SUMMARY
# TABLE 2.2 — ORDERED TRIAD SEGMENTATION SUMMARY
# Segments as columns; analytical measures as rows.
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path
import os
import tempfile

# ------------------------------------------------------------
# 1. VALIDATE ELIGIBLE TRIAD POPULATION
# ------------------------------------------------------------

# Source-row uniqueness was validated in the shared Chapter 2 setup cell.

eligible_triad_count_ch2 = eligible_triads_ch2.count()

eligible_fragrance_assignments_ch2 = (
    eligible_triads_ch2
    .agg(F.sum("fragrance_count").alias("n"))
    .first()["n"]
)

assert eligible_triad_count_ch2 > 0, (
    "No eligible triads were found. Review the source snapshot "
    "and eligibility rules."
)

assert eligible_fragrance_assignments_ch2 is not None, (
    "The eligible triad fragrance-assignment total is null."
)

assert eligible_fragrance_assignments_ch2 > 0, (
    "No fragrance assignments were found in the eligible triad population."
)

print("Chapter 2 population validation passed.")
print(f"Unique rated fragrance records: {source_rows_ch2:,}")
print(f"Eligible ordered triads: {eligible_triad_count_ch2:,}")
print(
    "Fragrances represented in eligible triads: "
    f"{eligible_fragrance_assignments_ch2:,}"
)


# ------------------------------------------------------------
# 2. PREPARE SHARED PANDAS DATASET
# ------------------------------------------------------------

triads_pdf_ch2 = eligible_triads_ch2.toPandas().copy()

for metric in [
    "fragrance_count",
    "mean_votes",
    "mean_rating",
    "share_rated_4_5_plus"
]:
    triads_pdf_ch2[metric] = pd.to_numeric(
        triads_pdf_ch2[metric],
        errors="coerce"
    )

triads_pdf_ch2 = triads_pdf_ch2.dropna(
    subset=[
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    ]
).copy()

assert not triads_pdf_ch2.empty, (
    "The cleaned triad dataset is empty. Review the metric columns "
    "and missing-value filtering."
)


# ------------------------------------------------------------
# 3. CALCULATE MEDIAN THRESHOLDS
# ------------------------------------------------------------

popularity_median_ch2 = (
    triads_pdf_ch2["mean_votes"].median()
)

rating_median_ch2 = (
    triads_pdf_ch2["mean_rating"].median()
)


# ------------------------------------------------------------
# 4. ASSIGN TRIAD SEGMENTS
# ------------------------------------------------------------

def assign_triad_segment_ch2(row):

    high_attention = (
        row["mean_votes"] >= popularity_median_ch2
    )

    high_rating = (
        row["mean_rating"] >= rating_median_ch2
    )

    if high_attention and high_rating:
        return "Standout Profiles"

    elif not high_attention and high_rating:
        return "Hidden Favourites"

    elif high_attention and not high_rating:
        return "Widely Explored"

    else:
        return "Quiet Profiles"


triads_pdf_ch2["segment"] = triads_pdf_ch2.apply(
    assign_triad_segment_ch2,
    axis=1
)


# ------------------------------------------------------------
# 5. SEGMENT ORDER AND COLOURS
# ------------------------------------------------------------

segment_order_ch2 = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

segment_colors_ch2 = {
    "Standout Profiles": BOOK_STYLE["accent"],
    "Hidden Favourites": BOOK_STYLE["women"],
    "Widely Explored": BOOK_STYLE["men"],
    "Quiet Profiles": BOOK_STYLE["unisex"]
}


# ------------------------------------------------------------
# 6. PRINT MEDIAN REFERENCE VALUES
# ------------------------------------------------------------

print(
    "\nMedian mean community votes per fragrance: "
    f"{popularity_median_ch2:,.2f}"
)

print(
    "Median mean fragrance rating: "
    f"{rating_median_ch2:.3f}"
)


# ------------------------------------------------------------
# 7. BUILD TRIAD SEGMENT SUMMARY
# ------------------------------------------------------------

segment_summary_ch2 = (
    triads_pdf_ch2
    .groupby("segment", observed=False)
    .agg(
        triad_count=("accord_triad", "count"),
        fragrance_assignments=("fragrance_count", "sum"),
        mean_votes_per_triad=("mean_votes", "mean"),
        mean_rating_per_triad=("mean_rating", "mean"),
        mean_triad_share_rated_4_5_plus=(
            "share_rated_4_5_plus",
            "mean"
        )
    )
    .reindex(segment_order_ch2)
    .reset_index()
)


# ------------------------------------------------------------
# 8. CREATE TRANSPOSED SUMMARY TABLE
# Segments are columns; measures are rows.
# ------------------------------------------------------------

segment_summary_transposed_ch2 = (
    segment_summary_ch2
    .set_index("segment")
    .reindex(segment_order_ch2)
    [
        [
            "triad_count",
            "fragrance_assignments",
            "mean_votes_per_triad",
            "mean_rating_per_triad",
            "mean_triad_share_rated_4_5_plus"
        ]
    ]
    .T
)

segment_summary_transposed_ch2.index = [
    "Number of ordered triads",
    "Fragrance assignments",
    "Mean votes per triad",
    "Mean rating per triad",
    "Share rated 4.5+"
]

segment_summary_transposed_ch2.index.name = "Measure"


# ------------------------------------------------------------
# 9. FORMAT VALUES
# ------------------------------------------------------------

for segment in segment_order_ch2:

    values = segment_summary_transposed_ch2[segment]

    formatted_values = []

    for row_index, value in enumerate(values):

        if pd.isna(value):
            formatted_values.append("—")

        elif row_index in [0, 1]:
            formatted_values.append(f"{value:,.0f}")

        elif row_index == 2:
            formatted_values.append(f"{value:,.1f}")

        elif row_index == 3:
            formatted_values.append(f"{value:.3f}")

        else:
            formatted_values.append(f"{value:.1%}")

    segment_summary_transposed_ch2[segment] = formatted_values


# Convert the index into the first display column.

segment_summary_display_ch2 = (
    segment_summary_transposed_ch2
    .reset_index()
)


# ------------------------------------------------------------
# 10. DEDICATED MATPLOTLIB TABLE RENDERER
# ------------------------------------------------------------

def render_triad_segment_table_ch2(table):

    data = table["data"].copy()

    fig, ax = plt.subplots(
        figsize=(15, 7),
        dpi=180
    )

    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    ax.set_position([0, 0, 1, 1])
    ax.axis("off")

    table_artist = ax.table(
        cellText=data.fillna("").astype(str).values.tolist(),
        colLabels=data.columns.tolist(),

        # Measure = 33%; four segment columns share 67%
        colWidths=[0.33] + [0.1675] * (len(data.columns) - 1),

        cellLoc="center",
        colLoc="center",

        bbox=[0.055, 0.10, 0.92, 0.69]
    )

    table_artist.auto_set_font_size(False)

    # Matching font size for column headers and table body
    matching_font_size = 15

    for (row, col), cell in table_artist.get_celld().items():

        cell.set_edgecolor(BOOK_STYLE["rule"])
        cell.set_linewidth(0.75)
        cell.PAD = 0.055

        if row == 0:

            cell.set_facecolor("#40332E")

            cell.set_text_props(
                color="#FFFDF9",
                weight="bold",
                fontsize=matching_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                ha="center",
                va="center"
            )

            cell.set_height(0.14)

        else:

            cell.set_facecolor(
                "#FFFDF9" if row % 2 else "#F0E9DF"
            )

            cell.set_height(0.12)

            if col == 0:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    weight="bold",
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="left",
                    va="center"
                )

            else:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="center",
                    va="center"
                )

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    fig.text(
        0.055,
        0.94,
        table["title"],
        color=BOOK_STYLE["text"],
        fontsize=20,
        fontweight="bold",
        ha="left",
        va="top"
    )

    # --------------------------------------------------------
    # Subtitle
    # --------------------------------------------------------

    subtitle = table.get("subtitle")

    if subtitle:

        fig.text(
            0.057,
            0.875,
            subtitle,
            color=BOOK_STYLE["muted"],
            fontsize=15,
            fontfamily=BOOK_TYPOGRAPHY["body"],
            ha="left",
            va="top"
        )

    return fig


# ------------------------------------------------------------
# 11. DISPLAY FUNCTION
# ------------------------------------------------------------

def show_triad_segment_table_ch2(table):

    fig = render_triad_segment_table_ch2(table)

    show_book_matplotlib(fig)

    plt.close(fig)


# ------------------------------------------------------------
# 12. PNG EXPORT FUNCTION
# ------------------------------------------------------------

def save_triad_segment_table_ch2(table, filename):

    output_dir = Path(
        "/Volumes/main/fragrance_analytics/reference_data/visualizations"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = output_dir / filename
    temp_path = None

    fig = render_triad_segment_table_ch2(table)

    try:

        with tempfile.NamedTemporaryFile(
            suffix=".png",
            dir=output_dir,
            delete=False
        ) as temp_file:

            temp_path = Path(temp_file.name)

        fig.savefig(
            temp_path,
            dpi=300,
            bbox_inches=None,
            pad_inches=0,
            facecolor=fig.get_facecolor()
        )

        os.replace(
            temp_path,
            output_path
        )

    finally:

        plt.close(fig)

        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

    print(f"Saved: {output_path}")


# ------------------------------------------------------------
# 13. CREATE, DISPLAY AND EXPORT TABLE 2.2
# ------------------------------------------------------------

triad_segment_table_ch2 = make_book_table(
    segment_summary_display_ch2,
    title="Ordered Triad Segmentation Summary",
    subtitle=(
        "Segment means are unweighted across triads. "
        "Fragrance assignments represent the sum of group-level "
        "triad fragrance counts."
    )
)

show_triad_segment_table_ch2(triad_segment_table_ch2)

save_triad_segment_table_ch2(
    triad_segment_table_ch2,
    "table_2_2_ordered_triad_segmentation_summary.png"
)


# ------------------------------------------------------------
# 14. DISPLAY TOP 20 TRIADS BY FRAGRANCE ASSIGNMENTS
# ------------------------------------------------------------

display(
    triads_pdf_ch2[
        [
            "accord_triad",
            "fragrance_count",
            "mean_votes",
            "mean_rating",
            "share_rated_4_5_plus",
            "segment"
        ]
    ]
    .sort_values(
        "fragrance_count",
        ascending=False
    )
    .head(20)
)

## 2.9 Figure 2.2 — Most Frequent Ordered Accord Triads

The chart ranks the 20 most frequent ordered triads by the number of fragrances assigned to each combination. Accord order is retained in every label.

In [0]:
# ============================================================
# FIGURE 2.2 — MOST FREQUENT ORDERED TRIADS
# ============================================================

def format_triad_label_ch2(value):
    parts = str(value).split(" + ")
    if len(parts) == 3:
        return f"{parts[0]} + {parts[1]}\n{parts[2]}"
    return str(value)

def plot_triad_ranking_ch2(
    metric,
    title,
    subtitle,
    filename,
    bar_color,
    value_format="number",
    top_n=20
):
    plot_df = (
        triads_pdf_ch2
        .dropna(subset=[metric])
        .nlargest(top_n, metric)
        .sort_values(metric, ascending=True)
        .copy()
    )

    plot_df["display_label"] = plot_df["accord_triad"].apply(
        format_triad_label_ch2
    )

    fig_height = max(8, len(plot_df) * 0.43 + 2.5)
    fig, ax = plt.subplots(figsize=(15, fig_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    y_positions = np.arange(len(plot_df))
    bars = ax.barh(
        y_positions,
        plot_df[metric],
        color=bar_color,
        edgecolor="none",
        height=0.68,
        zorder=3
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=BOOK_BAR_SIZES["category_pt"],
        color=BOOK_STYLE["text"]
    )

    max_value = float(plot_df[metric].max())
    label_offset = max(max_value * 0.012, 0.001)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        value = float(row[metric])
        if value_format == "rating":
            label = f"{value:.3f}"
        elif value_format == "percentage":
            label = f"{value:.1%}"
        elif value_format == "votes":
            label = f"{value:,.1f}"
        else:
            label = f"{value:,.0f}"

        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=BOOK_BAR_SIZES["value_pt"],
            color=BOOK_STYLE["text"]
        )

    ax.set_xlim(0, max_value * 1.16 if max_value > 0 else 1)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="y", length=0, labelsize=BOOK_BAR_SIZES["category_pt"], pad=7)
    ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
    ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    if value_format == "percentage":
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    elif value_format == "votes":
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    elif value_format == "rating":
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.2f}"))
    else:
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    for spine in ax.spines.values():
        spine.set_visible(False)

    apply_book_bar_layout(fig, bottom=0.08, right=0.96)
    add_book_figure_header(
        fig,
        title,
        subtitle,
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
    )

    fig.savefig(
        CHAPTER_2_OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches=None,
        facecolor=BOOK_STYLE["background"]
    )
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR / filename}")

plot_triad_ranking_ch2(
    metric="fragrance_count",
    title="The Most Frequent Ordered Accord Triads",
    subtitle="Twenty most frequent combinations; n denotes the number of fragrances",
    filename="figure_2_2_top_20_ordered_triads.png",
    bar_color=BOOK_STYLE["accent"],
    value_format="number"
)

## 2.10 Figure 2.4 — Community Attention and Mean Rating Across Ordered Triads

The bubble chart positions all eligible ordered triads by mean community votes per fragrance and mean fragrance rating. Bubble size represents the number of fragrances in the triad. Dashed reference lines mark the medians calculated across the full eligible population. Labels identify a selected subset of up to 35 triads for readability; they do not define the analytical population.

In [0]:
# ============================================================
# FIGURE 2.4 — ORDERED TRIAD ATTENTION / RATING SEGMENTATION
# All eligible triads are plotted; selected triads are labelled.
# ============================================================

from matplotlib.lines import Line2D
from adjustText import adjust_text

N_LARGEST_TRIAD_CH2 = 8
N_EXTREMES_TRIAD_CH2 = 6
MAX_LABELS_TRIAD_CH2 = 24
MIN_BUBBLE_TRIAD_CH2 = 55
MAX_BUBBLE_TRIAD_CH2 = 950

# ------------------------------------------------------------
# 1. CALCULATE BUBBLE SIZES
# ------------------------------------------------------------

triad_counts_ch2 = triads_pdf_ch2["fragrance_count"].astype(float)
triad_count_min_ch2 = triad_counts_ch2.min()
triad_count_max_ch2 = triad_counts_ch2.max()

def triad_bubble_size_ch2(count):
    if triad_count_max_ch2 == triad_count_min_ch2:
        return MIN_BUBBLE_TRIAD_CH2
    normalized = (
        (count - triad_count_min_ch2)
        / (triad_count_max_ch2 - triad_count_min_ch2)
    )
    normalized = max(0, min(1, normalized))
    return (
        MIN_BUBBLE_TRIAD_CH2
        + normalized ** 0.5
        * (MAX_BUBBLE_TRIAD_CH2 - MIN_BUBBLE_TRIAD_CH2)
    )

triads_pdf_ch2["bubble_size"] = triad_counts_ch2.apply(triad_bubble_size_ch2)

# ------------------------------------------------------------
# 2. SELECT TRIADS TO LABEL
# ------------------------------------------------------------

largest_triads_ch2 = triads_pdf_ch2.nlargest(
    N_LARGEST_TRIAD_CH2, "fragrance_count"
)

triad_labels_ch2 = pd.concat([
    triads_pdf_ch2.nlargest(N_EXTREMES_TRIAD_CH2, "mean_votes"),
    triads_pdf_ch2.nsmallest(N_EXTREMES_TRIAD_CH2, "mean_votes"),
    triads_pdf_ch2.nlargest(N_EXTREMES_TRIAD_CH2, "mean_rating"),
    triads_pdf_ch2.nsmallest(N_EXTREMES_TRIAD_CH2, "mean_rating"),
    largest_triads_ch2
]).drop_duplicates("accord_triad").copy()

triad_labels_ch2["is_largest"] = triad_labels_ch2["accord_triad"].isin(
    largest_triads_ch2["accord_triad"]
)
triad_labels_ch2["attention_extremeness"] = (
    (triad_labels_ch2["mean_votes"] - popularity_median_ch2).abs()
    / max(abs(popularity_median_ch2), 1)
)
triad_labels_ch2["rating_extremeness"] = (
    triad_labels_ch2["mean_rating"] - rating_median_ch2
).abs()

triad_labels_ch2 = (
    triad_labels_ch2
    .sort_values(
        ["is_largest", "attention_extremeness", "rating_extremeness"],
        ascending=[False, False, False]
    )
    .head(MAX_LABELS_TRIAD_CH2)
)

# ------------------------------------------------------------
# 3. DEFINE CHAPTER 1 SEGMENT COLOURS
# ------------------------------------------------------------

segment_colors_triads_ch2 = {
    "Standout Profiles": "#A66A4E",
    "Hidden Favourites": "#B58A9A",
    "Widely Explored": "#71818A",
    "Quiet Profiles": "#C5A85C"
}

# ------------------------------------------------------------
# 4. CREATE FIGURE
# ------------------------------------------------------------

fig_2_4_triads, ax = plt.subplots(figsize=(12, 10), dpi=180)
fig_2_4_triads.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

# ------------------------------------------------------------
# 5. PLOT ALL ELIGIBLE TRIADS
# ------------------------------------------------------------

for segment_name in segment_order_ch2:
    subset = triads_pdf_ch2[triads_pdf_ch2["segment"] == segment_name]
    if not subset.empty:
        ax.scatter(
            subset["mean_votes"],
            subset["mean_rating"],
            s=subset["bubble_size"],
            color=segment_colors_triads_ch2[segment_name],
            alpha=0.78,
            edgecolors=BOOK_STYLE["background"],
            linewidths=0.8,
            zorder=3
        )

# ------------------------------------------------------------
# 6. ADD TRIAD LABELS
# ------------------------------------------------------------

texts = []
for _, row in triad_labels_ch2.iterrows():
    label = (
        f"{format_triad_label_ch2(row['accord_triad'])} "
        f"(n={row['fragrance_count']:,.0f})"
    )
    text = ax.text(
        row["mean_votes"],
        row["mean_rating"],
        label,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["category_pt"] + 1,
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        alpha=0.98,
        zorder=5,
        bbox={
            "facecolor": BOOK_STYLE["background"],
            "edgecolor": "none",
            "alpha": 0.85,
            "pad": 1.5
        }
    )
    texts.append(text)

if texts:
    adjust_text(
        texts,
        ax=ax,
        expand=(1.2, 1.4),
        force_text=(0.5, 0.8),
        force_static=(0.5, 0.8),
        force_pull=(0.05, 0.1),
        ensure_inside_axes=True,
        iter_lim=500,
        arrowprops=dict(
            arrowstyle="-",
            color=BOOK_STYLE["muted"],
            lw=0.7,
            alpha=0.7
        )
    )

# ------------------------------------------------------------
# 7. ADD MEDIAN REFERENCE LINES
# ------------------------------------------------------------

ax.axvline(
    popularity_median_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)
ax.axhline(
    rating_median_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=2
)

# ------------------------------------------------------------
# 8. FORMAT AXES AND GRID
# ------------------------------------------------------------

ax.set_xlabel(
    "Mean community votes per fragrance",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)
ax.set_ylabel(
    "Mean fragrance rating",
    fontsize=BOOK_SIZES["category_pt"] + 1,
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"],
    labelpad=12
)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f"{y:.2f}"))
ax.tick_params(
    axis="both",
    length=0,
    labelsize=BOOK_SIZES["footer_pt"] + 1,
    pad=8,
    colors=BOOK_STYLE["text"]
)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="both",
    color=BOOK_STYLE["grid"],
    linewidth=0.7,
    alpha=0.9,
    zorder=0
)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# 9. LEGEND — RELATIVE SEGMENTS ONLY
# ------------------------------------------------------------

segment_handles_triads_ch2 = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=9,
        markerfacecolor=segment_colors_triads_ch2[name],
        markeredgecolor=BOOK_STYLE["background"],
        markeredgewidth=0.8,
        label=name
    )
    for name in segment_order_ch2
]

segment_legend_triads_ch2 = ax.legend(
    handles=segment_handles_triads_ch2,
    title="Relative segment",
    loc="upper right",
    bbox_to_anchor=(1.0, 1.0),
    frameon=True,
    facecolor=BOOK_STYLE["background"],
    edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["category_pt"] + 1,
    title_fontsize=BOOK_SIZES["category_pt"] + 2,
    labelcolor=BOOK_STYLE["text"],
    borderpad=0.9,
    labelspacing=0.75,
    handletextpad=0.8
)
segment_legend_triads_ch2.get_title().set_color(BOOK_STYLE["muted"])
segment_legend_triads_ch2.get_title().set_fontfamily(BOOK_TYPOGRAPHY["body"])

# ------------------------------------------------------------
# 10. ADD FIGURE HEADER AND APPLY LAYOUT
# ------------------------------------------------------------

add_book_figure_header(
    fig_2_4_triads,
    "Ordered Accord Triads: Community Attention and Mean Rating",
    "All eligible triads  |  Bubble size: number of fragrances  |  "
    "Labels show triad and fragrance count.",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)

fig_2_4_triads.subplots_adjust(
    left=0.085,
    right=0.985,
    top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_2_4_triads.get_figheight(),
    bottom=0.11
)

# ------------------------------------------------------------
# 11. RENDER AND SAVE
# ------------------------------------------------------------

fig_2_4_triads.savefig(
    CHAPTER_2_OUTPUT_DIR / "figure_2_4_triad_attention_rating_segmentation.png",
    dpi=300,
    bbox_inches=None,
    facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_2_4_triads)
plt.close(fig_2_4_triads)

print(f"Eligible triads plotted: {len(triads_pdf_ch2):,}")
print(f"Triads labelled: {len(triad_labels_ch2):,}")
print(f"Attention median: {popularity_median_ch2:.2f}")
print(f"Rating median: {rating_median_ch2:.3f}")
print(f"Saved: {CHAPTER_2_OUTPUT_DIR / 'figure_2_4_triad_attention_rating_segmentation.png'}")


## 2.11 Figures 2.9–2.12 — Highest-Rated Share Within Each Triad Segment

Each chart shows the 20 triads within one segment with the highest share of fragrances rated 4.5 or above. The metric complements mean rating; it is not interchangeable with it. Bar labels include the percentage and the number of fragrances in the triad.

In [0]:
# ============================================================
# FIGURES 2.9–2.12 — TOP TRIADS BY HIGH-RATING SHARE, BY SEGMENT
# ============================================================

def plot_triad_segment_ch2(segment_name, top_n=20):
    plot_df = (
        triads_pdf_ch2[
            triads_pdf_ch2["segment"] == segment_name
        ]
        .nlargest(top_n, "share_rated_4_5_plus")
        .sort_values("share_rated_4_5_plus", ascending=True)
        .copy()
    )

    if plot_df.empty:
        print(f"No triads found for segment: {segment_name}")
        return

    plot_df["display_label"] = plot_df["accord_triad"].apply(
        format_triad_label_ch2
    )

    fig_height = max(8, len(plot_df) * 0.43 + 2.5)
    fig, ax = plt.subplots(figsize=(15, fig_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    y_positions = np.arange(len(plot_df))
    bars = ax.barh(
        y_positions,
        plot_df["share_rated_4_5_plus"],
        color=segment_colors_ch2[segment_name],
        edgecolor="none",
        height=0.68,
        zorder=3
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=BOOK_BAR_SIZES["category_pt"],
        color=BOOK_STYLE["text"]
    )

    max_value = float(plot_df["share_rated_4_5_plus"].max())
    label_offset = max(max_value * 0.012, 0.001)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        value = float(row["share_rated_4_5_plus"])
        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1%}  (n={int(row['fragrance_count']):,})",
            va="center",
            ha="left",
            fontsize=BOOK_BAR_SIZES["value_pt"],
            color=BOOK_STYLE["text"]
        )

    ax.set_xlim(0, max_value * 1.42 if max_value > 0 else 1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel(
        "Share of fragrances rated 4.5+",
        fontsize=BOOK_SIZES["category_pt"],
        color=BOOK_STYLE["muted"],
        labelpad=10
    )
    ax.set_ylabel("")
    ax.tick_params(axis="y", length=0, labelsize=BOOK_BAR_SIZES["category_pt"], pad=7)
    ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
    ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    apply_book_bar_layout(fig, bottom=0.08, right=0.96)
    add_book_figure_header(
        fig,
        f"{segment_name}: Triads with the Highest 4.5+ Share",
        "Top 20 ordered triads by share of fragrances rated 4.5 or above",
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
    )

    figure_no = {"Standout Profiles": 9, "Hidden Favourites": 10, "Widely Explored": 11, "Quiet Profiles": 12}[segment_name]
    filename = f"figure_2_{figure_no}_triad_{segment_name.lower().replace(chr(32), chr(95))}.png"
    fig.savefig(
        CHAPTER_2_OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches=None,
        facecolor=BOOK_STYLE["background"]
    )
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR / filename}")

for segment_name in segment_order_ch2:
    plot_triad_segment_ch2(segment_name, top_n=20)

## 2.12 Dyad and triad comparison

The following comparison brings the segment summaries together. Segment-level means remain unweighted across combinations; fragrance assignments are summed group-level counts. The dyad and triad thresholds are independent and their eligible populations differ.

In [0]:
# ============================================================
# TABLE 2.3 — SEPARATE DYAD AND TRIAD COMPARISONS BY SEGMENT
# Four independent publication-ready visualizations.
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import os
import tempfile


# ------------------------------------------------------------
# 1. PREPARE DYAD AND TRIAD DATA
# ------------------------------------------------------------

pair_compare_ch2 = pair_segment_summary_ch2.rename(columns={
    "pair_count": "dyad_count",
    "fragrance_assignments": "dyad_fragrance_assignments",
    "mean_votes_per_pair": "dyad_mean_votes",
    "mean_rating_per_pair": "dyad_mean_rating",
    "mean_pair_share_rated_4_5_plus": "dyad_mean_share_4_5_plus"
})

triad_compare_ch2 = segment_summary_ch2.rename(columns={
    "triad_count": "triad_count",
    "fragrance_assignments": "triad_fragrance_assignments",
    "mean_votes_per_triad": "triad_mean_votes",
    "mean_rating_per_triad": "triad_mean_rating",
    "mean_triad_share_rated_4_5_plus": "triad_mean_share_4_5_plus"
})

dyad_triad_comparison_ch2 = pair_compare_ch2.merge(
    triad_compare_ch2,
    on="segment",
    how="outer"
)

comparison_index_ch2 = (
    dyad_triad_comparison_ch2.set_index("segment")
)


# ------------------------------------------------------------
# 2. SEGMENT ORDER AND METRIC DEFINITIONS
# ------------------------------------------------------------

segment_order_ch2 = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

available_segments_ch2 = (
    dyad_triad_comparison_ch2["segment"]
    .dropna()
    .tolist()
)

segment_order_ch2 = [
    segment for segment in segment_order_ch2
    if segment in available_segments_ch2
] + [
    segment for segment in available_segments_ch2
    if segment not in segment_order_ch2
]

comparison_metrics_ch2 = [
    (
        "Number of combinations",
        "dyad_count",
        "triad_count",
        "count"
    ),
    (
        "Fragrance assignments",
        "dyad_fragrance_assignments",
        "triad_fragrance_assignments",
        "count"
    ),
    (
        "Mean votes per combination",
        "dyad_mean_votes",
        "triad_mean_votes",
        "votes"
    ),
    (
        "Mean rating per combination",
        "dyad_mean_rating",
        "triad_mean_rating",
        "rating"
    ),
    (
        "Share rated 4.5+",
        "dyad_mean_share_4_5_plus",
        "triad_mean_share_4_5_plus",
        "share"
    )
]


# ------------------------------------------------------------
# 3. VALUE FORMATTING
# ------------------------------------------------------------

def format_comparison_value_ch2(value, value_format):

    if pd.isna(value):
        return "—"

    if value_format == "count":
        return f"{value:,.0f}"

    if value_format == "votes":
        return f"{value:,.1f}"

    if value_format == "rating":
        return f"{value:.3f}"

    if value_format == "share":
        return f"{value:.1%}"

    return str(value)


# ------------------------------------------------------------
# 4. OUTPUT DIRECTORY
# ------------------------------------------------------------

output_dir_ch2 = Path(
    "/Volumes/main/fragrance_analytics/reference_data/visualizations"
)

output_dir_ch2.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. SHARED TABLE RENDERER
# ------------------------------------------------------------

def render_segment_comparison_ch2(segment, segment_table_data):

    fig, ax = plt.subplots(
        figsize=(13, 7),
        dpi=180
    )

    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    ax.set_position([0, 0, 1, 1])
    ax.axis("off")

    table_artist = ax.table(
        cellText=segment_table_data.fillna("").astype(str).values.tolist(),
        colLabels=segment_table_data.columns.tolist(),

        # Measure = 33%; Dyads and Triads share the remaining 67%
        colWidths=[0.33, 0.335, 0.335],

        cellLoc="center",
        colLoc="center",

        bbox=[0.065, 0.10, 0.87, 0.69]
    )

    table_artist.auto_set_font_size(False)

    # Matching font size for headers and all body cells
    matching_font_size = 15

    for (row, col), cell in table_artist.get_celld().items():

        cell.set_edgecolor(BOOK_STYLE["rule"])
        cell.set_linewidth(0.75)

        # Compact internal spacing
        cell.PAD = 0.055

        if row == 0:

            cell.set_facecolor("#40332E")

            cell.set_text_props(
                color="#FFFDF9",
                weight="bold",
                fontsize=matching_font_size,
                fontfamily=BOOK_TYPOGRAPHY["body"],
                ha="center",
                va="center"
            )

            # Compact header height
            cell.set_height(0.09)

        else:

            cell.set_facecolor(
                "#FFFDF9" if row % 2 else "#F0E9DF"
            )

            # Compact body rows
            cell.set_height(0.075)

            if col == 0:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    weight="bold",
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="left",
                    va="center"
                )

            else:

                cell.set_text_props(
                    color=BOOK_STYLE["text"],
                    fontsize=matching_font_size,
                    fontfamily=BOOK_TYPOGRAPHY["body"],
                    ha="center",
                    va="center"
                )

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    fig.text(
        0.065,
        0.94,
        f"{segment} — Dyad and Triad Comparison",
        ha="left",
        va="top",
        color=BOOK_STYLE["text"],
        fontsize=21,
        fontweight="bold"
    )

    # --------------------------------------------------------
    # SUBTITLE
    # --------------------------------------------------------

    fig.text(
        0.067,
        0.865,
        (
            "Comparison of combination counts, fragrance assignments, "
            "platform attention and rating indicators."
        ),
        ha="left",
        va="top",
        color=BOOK_STYLE["muted"],
        fontsize=15,
        fontfamily=BOOK_TYPOGRAPHY["body"]
    )

    return fig


# ------------------------------------------------------------
# 6. DISPLAY FUNCTION
# ------------------------------------------------------------

def show_segment_comparison_ch2(segment, segment_table_data):

    fig = render_segment_comparison_ch2(
        segment,
        segment_table_data
    )

    show_book_matplotlib(fig)

    plt.close(fig)


# ------------------------------------------------------------
# 7. PNG EXPORT FUNCTION
# ------------------------------------------------------------

def save_segment_comparison_ch2(
    segment,
    segment_table_data,
    filename
):

    output_path = output_dir_ch2 / filename
    temp_path = None

    fig = render_segment_comparison_ch2(
        segment,
        segment_table_data
    )

    try:

        with tempfile.NamedTemporaryFile(
            suffix=".png",
            dir=output_dir_ch2,
            delete=False
        ) as temp_file:

            temp_path = Path(temp_file.name)

        fig.savefig(
            temp_path,
            format="png",
            dpi=300,
            bbox_inches=None,
            pad_inches=0,
            facecolor=fig.get_facecolor(),
            edgecolor=fig.get_facecolor()
        )

        assert (
            temp_path.is_file()
            and temp_path.stat().st_size > 0
        ), f"PNG export failed: {temp_path}"

        os.replace(
            temp_path,
            output_path
        )

    finally:

        plt.close(fig)

        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

    print(
        f"Saved: {output_path} "
        f"({output_path.stat().st_size:,} bytes)"
    )


# ------------------------------------------------------------
# 8. CREATE ONE TABLE PER SEGMENT
# ------------------------------------------------------------

segment_filename_map_ch2 = {
    "Standout Profiles": "table_2_3a_standout_profiles.png",
    "Hidden Favourites": "table_2_3b_hidden_favourites.png",
    "Widely Explored": "table_2_3c_widely_explored.png",
    "Quiet Profiles": "table_2_3d_quiet_profiles.png"
}

segment_visualizations_ch2 = {}

for segment in segment_order_ch2:

    segment_rows_ch2 = []

    for (
        metric_label,
        dyad_column,
        triad_column,
        value_format
    ) in comparison_metrics_ch2:

        dyad_value = (
            comparison_index_ch2.loc[segment, dyad_column]
            if dyad_column in comparison_index_ch2.columns
            else None
        )

        triad_value = (
            comparison_index_ch2.loc[segment, triad_column]
            if triad_column in comparison_index_ch2.columns
            else None
        )

        segment_rows_ch2.append({
            "Measure": metric_label,
            "Dyads": format_comparison_value_ch2(
                dyad_value,
                value_format
            ),
            "Triads": format_comparison_value_ch2(
                triad_value,
                value_format
            )
        })

    segment_table_data_ch2 = pd.DataFrame(
        segment_rows_ch2
    )

    segment_visualizations_ch2[segment] = (
        segment_table_data_ch2
    )

    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    show_segment_comparison_ch2(
        segment,
        segment_table_data_ch2
    )

    # --------------------------------------------------------
    # EXPORT PNG AT 300 DPI
    # --------------------------------------------------------

    filename_ch2 = segment_filename_map_ch2[segment]

    save_segment_comparison_ch2(
        segment,
        segment_table_data_ch2,
        filename_ch2
    )


# ------------------------------------------------------------
# 9. MEDIAN REFERENCE VALUES
# ------------------------------------------------------------

print("\nDYAD MEDIANS")
print("Attention:", round(popularity_median_pairs_ch2, 3))
print("Rating:", round(rating_median_pairs_ch2, 4))

print("\nTRIAD MEDIANS")
print("Attention:", round(popularity_median_ch2, 3))
print("Rating:", round(rating_median_ch2, 4))

print("\nFour segment comparison tables prepared and exported.")

## 2.13 Reproducibility and output validation

- Source table: `main.fragrance_analytics.fragrantica_perfumes`.
- Accord positions are preserved in their original order.
- Dyad aggregation uses a minimum support of 30 fragrances; segmentation uses at least 100 (147 eligible dyads in the validated source snapshot).
- Triad segmentation uses at least 50 fragrances (159 eligible triads and 16,073 fragrance assignments in the validated source snapshot).
- Dyad and triad medians are calculated independently.
- Segment summary metrics are unweighted means across combinations.
- Segment-specific charts rank by the share of associated fragrances rated 4.5 or above; this measure does not define segment membership.
- PNG files are saved at 300 dpi.
- Community votes represent platform activity, not sales, market share or unique consumers.

**Output location:** `/Volumes/main/fragrance_analytics/reference_data/visualizations/`.

In [0]:
# ============================================================
# CHAPTER 2 — FINAL FIGURE AND TABLE EXPORT VALIDATION
# Run after all figure- and table-generation cells have completed.
# ============================================================

expected_chapter_2_pngs = [
    'figure_2_1_top_20_ordered_dyads.png',
    'figure_2_2_top_20_ordered_triads.png',
    'figure_2_3_dyad_attention_rating_segmentation.png',
    'figure_2_4_triad_attention_rating_segmentation.png',
    'figure_2_5_dyad_standout_profiles.png',
    'figure_2_6_dyad_hidden_favourites.png',
    'figure_2_7_dyad_widely_explored.png',
    'figure_2_8_dyad_quiet_profiles.png',
    'figure_2_9_triad_standout_profiles.png',
    'figure_2_10_triad_hidden_favourites.png',
    'figure_2_11_triad_widely_explored.png',
    'figure_2_12_triad_quiet_profiles.png'
]

expected_chapter_2_table_pngs = [
    'table_2_1_ordered_dyad_segmentation_summary.png',
    'table_2_2_ordered_triad_segmentation_summary.png',
    'table_2_3_dyad_triad_segment_comparison.png'
]

missing_chapter_2_pngs = [
    name for name in expected_chapter_2_pngs
    if not (CHAPTER_2_OUTPUT_DIR / name).is_file()
]
missing_chapter_2_table_pngs = [
    name for name in expected_chapter_2_table_pngs
    if not (CHAPTER_2_OUTPUT_DIR / name).is_file()
]

assert not missing_chapter_2_pngs, (
    'Chapter 2 figure export validation failed. Missing files: '
    + ', '.join(missing_chapter_2_pngs)
)
assert not missing_chapter_2_table_pngs, (
    'Chapter 2 table export validation failed. Missing files: '
    + ', '.join(missing_chapter_2_table_pngs)
)

print(f'Chapter 2 figure exports verified: {len(expected_chapter_2_pngs)} PNG files')
print(f'Chapter 2 table exports verified: {len(expected_chapter_2_table_pngs)} PNG files')
for name in expected_chapter_2_pngs + expected_chapter_2_table_pngs:
    print(CHAPTER_2_OUTPUT_DIR / name)